# Setting config


In [4]:
# import lib python
import pandas as pd
from datetime import datetime
from sqlalchemy import create_engine

# immport utils ca nhan
from utils.config import *
from utils.db import *

# import log
import logging
import os

# Log

In [5]:
# Log setting

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%d-%m_%Y %H:%M:%S',
    handlers=[
        logging.FileHandler(
            f'log/ETL_Project_{datetime.now():%Y%m%d}.log'
        ),
        logging.StreamHandler()
    ]
)

log = logging.getLogger(__name__)

# Extract

In [6]:
def extract():
    log.info("Extract started!")

    df_d_center = pd.read_csv('rawdata/distribution_centers.csv')
    log.info(f'Get data distribution_centers done: Total {len(df_d_center)} rows')

    df_i_items = pd.read_csv('rawdata/inventory_items.csv')
    log.info(f'Get data inventory_items done: Total {len(df_i_items)} rows')

    df_o_items = pd.read_csv('rawdata/order_items.csv')
    log.info(f'Get data order_items done: Total {len(df_o_items)} rows')

    df_orders = pd.read_csv('rawdata/orders.csv')
    log.info(f'Get data orders done: Total {len(df_orders)} rows')

    df_products = pd.read_csv('rawdata/products.csv')
    log.info(f'Get data products done: Total {len(df_products)} rows')

    df_users = pd.read_csv('rawdata/users.csv')
    log.info(f'Get data users done: Total {len(df_users)} rows')

    df_events = pd.read_csv('rawdata/events.csv')
    log.info(f'Get data events done: Total {len(df_events)} rows')

    return df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users

# Transform

In [13]:
df_users.head(10)

,id,first_name,last_name,email,age,gender,state,street_address,postal_code,city,country,latitude,longitude,traffic_source,created_at
0,457,Timothy,Bush,timothybush@example.net,65,M,Acre,87620 Johnson Hills,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2022-07-19 13:51:00+00:00
1,6578,Elizabeth,Martinez,elizabethmartinez@example.com,34,F,Acre,1705 Nielsen Land,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2023-11-08 18:49:00+00:00
2,36280,Christopher,Mendoza,christophermendoza@example.net,13,M,Acre,125 Turner Isle Apt. 264,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Email,2019-08-24 06:10:00+00:00
3,60193,Jimmy,Conner,jimmyconner@example.com,64,M,Acre,0966 Jose Branch Apt. 008,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2020-02-15 11:26:00+00:00
4,64231,Natasha,Wilson,natashawilson@example.net,25,F,Acre,20798 Phillip Trail Apt. 392,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2020-03-13 06:45:00+00:00
5,72187,Andrea,Bryant,andreabryant@example.org,47,F,Acre,622 Sims Field,69917-400,Rio Branco,Brasil,-9.945568,-67.835610,Search,2022-05-27 14:53:00+00:00
6,22528,Keith,Barnett,keithbarnett@example.com,64,M,Acre,53908 Amy Fork Apt. 281,69940-000,Sena Madureira,Brasil,-9.857324,-69.437057,Search,2019-03-21 10:58:00+00:00
7,34417,Joanne,Mcpherson,joannemcpherson@example.com,28,F,Acre,53907 John Unions Apt. 099,69940-000,Sena Madureira,Brasil,-9.857324,-69.437057,Search,2024-01-12 19:02:31.096664+00:00
8,50098,Ian,Burke,ianburke@example.net,52,M,Acre,91902 Andrea Stream Apt. 563,69940-000,Sena Madureira,Brasil,-9.857324,-69.437057,Email,2019-04-12 06:11:00+00:00
9,63352,David,Orozco,davidorozco@example.org,18,M,Acre,9576 Dominique Run,69940-000,Sena Madureira,Brasil,-9.857324,-69.437057,Search,2023-10-29 12:33:00+00:00


In [12]:
df_products.head(10)

,id,cost,category,name,brand,retail_price,department,sku,distribution_center_id
0,13842,2.51875,Accessories,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,EBD58B8A3F1D72F4206201DA62FB1204,1
1,13928,2.33835,Accessories,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,2EAC42424D12436BDD6A5B8A88480CC3,1
2,14115,4.87956,Accessories,Enzyme Regular Solid Army Caps-Black W35S45D,MG,10.99,Women,EE364229B2791D1EF9355708EFF0BA34,1
3,14157,4.64877,Accessories,Enzyme Regular Solid Army Caps-Olive W35S45D (...,MG,10.99,Women,00BD13095D06C20B11A2993CA419D16B,1
4,14273,6.50793,Accessories,Washed Canvas Ivy Cap - Black W11S64C,MG,15.99,Women,F531DC20FDE20B7ADF3A73F52B71D0AF,1
5,15674,3.10625,Plus,Low Profile Dyed Cotton Twill Cap - Navy W39S55D,MG,6.25,Women,63894CE404B8C652915C41EF8B879D20,1
6,15816,3.17730,Plus,Low Profile Dyed Cotton Twill Cap - Putty W39S55D,MG,5.95,Women,151EA8C2D98CE89C2336324C11B1E107,1
7,28646,8.73563,Accessories,4 Panel Large Bill Flap Hat W15S48B (One Size ...,MG,19.99,Men,789334DE6DAA80D83AB4ACB6A4BF5AC7,1
8,28670,2.67594,Accessories,Low Profile Dyed Cotton Twill Cap - Black W39S55D,MG,6.18,Men,E74843B99DA8B29775C6AA9080436844,1
9,28714,2.27500,Accessories,Low Profile Dyed Cotton Twill Cap - Khaki W39S55D,MG,6.25,Men,8CA33D44648CC9FEECEF21E5A7123291,1


In [8]:
df_d_center.head(10)

,id,name,latitude,longitude
0,1,Memphis TN,35.1174,-89.9711
1,2,Chicago IL,41.8369,-87.6847
2,3,Houston TX,29.7604,-95.3698
3,4,Los Angeles CA,34.0500,-118.2500
4,5,New Orleans LA,29.9500,-90.0667
5,6,Port Authority of New York/New Jersey NY/NJ,40.6340,-73.7834
6,7,Philadelphia PA,39.9500,-75.1667
7,8,Mobile AL,30.6944,-88.0431
8,9,Charleston SC,32.7833,-79.9333
9,10,Savannah GA,32.0167,-81.1167


In [9]:
df_i_items.head(10)

,id,product_id,created_at,sold_at,cost,product_category,product_name,product_brand,product_retail_price,product_department,product_sku,product_distribution_center_id
0,67971,13844,2022-07-02 07:09:20+00:00,2022-07-24 06:33:20+00:00,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
1,67972,13844,2023-12-20 03:28:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
2,67973,13844,2023-06-04 02:53:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
3,72863,13844,2021-10-16 22:58:52+00:00,2021-11-22 02:19:52+00:00,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
4,72864,13844,2021-08-07 16:33:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
5,101826,13844,2023-10-22 23:08:09+00:00,2023-11-15 13:49:09+00:00,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
6,101827,13844,2023-10-10 08:15:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
7,107797,13844,2023-12-27 05:50:59+00:00,2024-01-13 23:58:59+00:00,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
8,107798,13844,2020-12-06 18:13:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7
9,107799,13844,2020-01-30 04:46:00+00:00,NaN,2.76804,Accessories,(ONE) 1 Satin Headband,Funny Girl Designs,6.99,Women,2A3E953A5E3D81E67945BCE5519F84C8,7


In [10]:
df_o_items.head(10)

,id,order_id,user_id,product_id,inventory_item_id,status,created_at,shipped_at,delivered_at,returned_at,sale_price
0,152013,104663,83582,14235,410368,Cancelled,2023-05-07 06:08:40+00:00,NaN,NaN,NaN,0.02
1,40993,28204,22551,14235,110590,Complete,2023-03-14 03:47:21+00:00,2023-03-15 22:57:00+00:00,2023-03-18 01:08:00+00:00,NaN,0.02
2,51224,35223,28215,14235,138236,Complete,2023-12-05 13:25:30+00:00,2023-12-06 01:20:00+00:00,2023-12-10 10:04:00+00:00,NaN,0.02
3,36717,25278,20165,14235,99072,Shipped,2023-12-22 20:48:19+00:00,2023-12-24 16:44:00+00:00,NaN,NaN,0.02
4,131061,90241,71954,14235,353798,Shipped,2022-06-19 16:57:59+00:00,2022-06-19 19:29:00+00:00,NaN,NaN,0.02
5,154022,106052,84693,14235,415780,Shipped,2023-09-19 09:13:08+00:00,2023-09-16 09:24:00+00:00,NaN,NaN,0.02
6,67253,46269,37023,14159,181497,Complete,2021-10-31 01:26:46+00:00,2021-11-01 17:47:00+00:00,2021-11-02 13:29:00+00:00,NaN,0.49
7,116116,80002,63821,14159,313368,Complete,2022-04-01 13:25:52+00:00,2022-03-31 05:13:00+00:00,2022-04-02 22:18:00+00:00,NaN,0.49
8,28239,19512,15553,14159,76146,Shipped,2023-10-29 08:08:40+00:00,2023-10-29 16:59:00+00:00,NaN,NaN,0.49
9,69641,47894,38312,14159,187914,Shipped,2024-01-07 23:06:39+00:00,2024-01-10 19:32:00+00:00,NaN,NaN,0.49


In [11]:
df_orders.head()

,order_id,user_id,status,gender,created_at,returned_at,shipped_at,delivered_at,num_of_item
0,8,5,Cancelled,F,2022-10-20 10:03:00+00:00,NaN,NaN,NaN,3
1,60,44,Cancelled,F,2023-01-20 02:12:00+00:00,NaN,NaN,NaN,1
2,64,46,Cancelled,F,2021-12-06 09:11:00+00:00,NaN,NaN,NaN,1
3,89,65,Cancelled,F,2020-08-13 09:58:00+00:00,NaN,NaN,NaN,1
4,102,76,Cancelled,F,2023-01-17 08:17:00+00:00,NaN,NaN,NaN,2


# Load

# Main execute

In [7]:
df_d_center, df_events, df_i_items, df_o_items, df_orders, df_products, df_users = extract()

29-10_2026 21:52:32 | INFO | Extract started!
29-10_2026 21:52:32 | INFO | Get data distribution_centers done: Total 10 rows
29-10_2026 21:52:33 | INFO | Get data inventory_items done: Total 490705 rows
29-10_2026 21:52:33 | INFO | Get data order_items done: Total 181759 rows
29-10_2026 21:52:33 | INFO | Get data orders done: Total 125226 rows
29-10_2026 21:52:33 | INFO | Get data products done: Total 29120 rows
29-10_2026 21:52:33 | INFO | Get data users done: Total 100000 rows
29-10_2026 21:52:38 | INFO | Get data events done: Total 2431963 rows
